In [ ]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# Loading the datasets
data_2020 = pd.read_csv('/content/elections_results_2020.csv')
data_2022 = pd.read_csv('/content/elections_results_2022.csv')

# Displaying the first 5 rows of the dataset
display(data_2020.head())
display(data_2022.head())

,countyname,party_dem,party_rep,win_margin_abs,win_margin_pct,winning_party
0,ADAMS,18207,37523,19316,34.659968,Republican
1,ALLEGHENY,429065,282324,146741,20.627392,Democrat
2,ARMSTRONG,8457,27489,19032,52.946086,Republican
3,BEAVER,38122,54759,16637,17.912167,Republican
4,BEDFORD,4367,23025,18658,68.114778,Republican


,countyname,party_dem,party_rep,win_margin_abs,win_margin_pct,winning_party
0,BERKS,104695,105415,720,0.342678,Republican
1,BLAIR,19594,46350,26756,40.573820,Republican
2,ADAMS,17879,38722,20843,36.824438,Republican
3,ALLEGHENY,497272,246412,250860,33.732069,Democrat
4,ARMSTRONG,11051,24268,13217,37.421784,Republican


In [3]:
# Merge the two datasets on a common key like 'countyname'
merged_results = pd.merge(data_2020, data_2022, on='countyname', suffixes=('_2020', '_2022'))

# Calculate the shifts for each party by subtracting 2020 results from 2022 results
merged_results['shift_party_dem'] = merged_results['party_dem_2022'] - merged_results['party_dem_2020']
merged_results['shift_party_rep'] = merged_results['party_rep_2022'] - merged_results['party_rep_2020']

# Optionally calculate the percentage shifts relative to 2020 totals
merged_results['shift_party_dem_pct'] = (merged_results['shift_party_dem'] / merged_results['party_dem_2020']) * 100
merged_results['shift_party_rep_pct'] = (merged_results['shift_party_rep'] / merged_results['party_rep_2020']) * 100

# Display the results with the calculated shifts
display(merged_results.head())

,countyname,party_dem_2020,party_rep_2020,win_margin_abs_2020,win_margin_pct_2020,winning_party_2020,party_dem_2022,party_rep_2022,win_margin_abs_2022,win_margin_pct_2022,winning_party_2022,shift_party_dem,shift_party_rep,shift_party_dem_pct,shift_party_rep_pct
0,ADAMS,18207,37523,19316,34.659968,Republican,17879,38722,20843,36.824438,Republican,-328,1199,-1.801505,3.195374
1,ALLEGHENY,429065,282324,146741,20.627392,Democrat,497272,246412,250860,33.732069,Democrat,68207,-35912,15.896659,-12.720137
2,ARMSTRONG,8457,27489,19032,52.946086,Republican,11051,24268,13217,37.421784,Republican,2594,-3221,30.672815,-11.717414
3,BEAVER,38122,54759,16637,17.912167,Republican,47425,45575,1850,1.989247,Democrat,9303,-9184,24.403232,-16.771672
4,BEDFORD,4367,23025,18658,68.114778,Republican,5766,22799,17033,59.628917,Republican,1399,-226,32.035722,-0.981542


In [4]:
# Merge the datasets based on the county names
merged_results = pd.merge(data_2020, data_2022, on='countyname', suffixes=('_2020', '_2022'))

# Calculate shifts for each party
shift_dem_abs = merged_results['party_dem_2022'] - merged_results['party_dem_2020']
shift_rep_abs = merged_results['party_rep_2022'] - merged_results['party_rep_2020']

# Calculate percentage shifts relative to the 2020 totals
shift_dem_pct = (shift_dem_abs / merged_results['party_dem_2020']) * 100
shift_rep_pct = (shift_rep_abs / merged_results['party_rep_2020']) * 100

# Organize the results in a new DataFrame
shift_results = pd.DataFrame({
    'countyname': merged_results['countyname'],
    'party_dem_2020': merged_results['party_dem_2020'],
    'party_dem_2022': merged_results['party_dem_2022'],
    'shift_party_dem_abs': shift_dem_abs,
    'shift_party_dem_pct': shift_dem_pct,
    'party_rep_2020': merged_results['party_rep_2020'],
    'party_rep_2022': merged_results['party_rep_2022'],
    'shift_party_rep_abs': shift_rep_abs,
    'shift_party_rep_pct': shift_rep_pct
})

# Round the percentage columns for a cleaner view
shift_results[['shift_party_dem_pct', 'shift_party_rep_pct']] = shift_results[['shift_party_dem_pct', 'shift_party_rep_pct']].round(2)

# Sort the results for easier interpretation
shift_results = shift_results.sort_values(by='countyname').reset_index(drop=True)

# Display the table for the shifts
display(shift_results)

,countyname,party_dem_2020,party_dem_2022,shift_party_dem_abs,shift_party_dem_pct,party_rep_2020,party_rep_2022,shift_party_rep_abs,shift_party_rep_pct
0,ADAMS,18207,17879,-328,-1.80,37523,38722,1199,3.20
1,ALLEGHENY,429065,497272,68207,15.90,282324,246412,-35912,-12.72
2,ARMSTRONG,8457,11051,2594,30.67,27489,24268,-3221,-11.72
3,BEAVER,38122,47425,9303,24.40,54759,45575,-9184,-16.77
4,BEDFORD,4367,5766,1399,32.04,23025,22799,-226,-0.98
...,...,...,...,...,...,...,...,...,...
61,WASHINGTON,45088,55369,10281,22.80,72080,64409,-7671,-10.64
62,WAYNE,9191,8582,-609,-6.63,18637,18904,267,1.43
63,WESTMORELAND,72129,90369,18240,25.29,130218,117710,-12508,-9.61
64,WYOMING,4704,4756,52,1.11,9936,10280,344,3.46


In [5]:

# binary encoding (0 for Republican and 1 for Democrat)

# Determine and encode the winning party directly in the columns
merged_results['winning_party_2020'] = merged_results.apply(lambda x: 1 if x['party_dem_2020'] > x['party_rep_2020'] else 0, axis=1)
merged_results['winning_party_2022'] = merged_results.apply(lambda x: 1 if x['party_dem_2022'] > x['party_rep_2022'] else 0, axis=1)

# Organize the results in a new DataFrame with all the previous and adjusted columns
shift_results = pd.DataFrame({
    'countyname': merged_results['countyname'],
    'party_dem_2020': merged_results['party_dem_2020'],
    'party_dem_2022': merged_results['party_dem_2022'],
    'shift_party_dem_points': shift_dem_abs,
    'shift_party_dem_pct': shift_dem_pct,
    'party_rep_2020': merged_results['party_rep_2020'],
    'party_rep_2022': merged_results['party_rep_2022'],
    'shift_party_rep_points': shift_rep_abs,
    'shift_party_rep_pct': shift_rep_pct,
    'winning_party_2020': merged_results['winning_party_2020'],  # Binary encoding directly
    'winning_party_2022': merged_results['winning_party_2022'],  # Binary encoding directly
    'winning_party_shift': merged_results.apply(
        lambda x: 'No Shift' if x['winning_party_2020'] == x['winning_party_2022'] else f"{x['winning_party_2020']} to {x['winning_party_2022']}",
        axis=1
    )
})

# Round the percentage columns for a cleaner view
shift_results[['shift_party_dem_pct', 'shift_party_rep_pct']] = shift_results[['shift_party_dem_pct', 'shift_party_rep_pct']].round(2)

# Sorting the results for easier interpretation
shift_results = shift_results.sort_values(by='countyname').reset_index(drop=True)

# Displaying the table for the shifts
display(shift_results)

# Saving to a CSV file
shift_results.to_csv('elections_shifts_2020-22.csv', index=False)

,countyname,party_dem_2020,party_dem_2022,shift_party_dem_points,shift_party_dem_pct,party_rep_2020,party_rep_2022,shift_party_rep_points,shift_party_rep_pct,winning_party_2020,winning_party_2022,winning_party_shift
0,ADAMS,18207,17879,-328,-1.80,37523,38722,1199,3.20,0,0,No Shift
1,ALLEGHENY,429065,497272,68207,15.90,282324,246412,-35912,-12.72,1,1,No Shift
2,ARMSTRONG,8457,11051,2594,30.67,27489,24268,-3221,-11.72,0,0,No Shift
3,BEAVER,38122,47425,9303,24.40,54759,45575,-9184,-16.77,0,1,0 to 1
4,BEDFORD,4367,5766,1399,32.04,23025,22799,-226,-0.98,0,0,No Shift
...,...,...,...,...,...,...,...,...,...,...,...,...
61,WASHINGTON,45088,55369,10281,22.80,72080,64409,-7671,-10.64,0,0,No Shift
62,WAYNE,9191,8582,-609,-6.63,18637,18904,267,1.43,0,0,No Shift
63,WESTMORELAND,72129,90369,18240,25.29,130218,117710,-12508,-9.61,0,0,No Shift
64,WYOMING,4704,4756,52,1.11,9936,10280,344,3.46,0,0,No Shift
